# 🚗 Notebook 2: Delivery Routing and ETA Estimation

When a customer opens the Gopuff app, the system needs to figure out **which warehouses can deliver to them** and **how long it will take**. This is harder than it sounds — you can’t just pick the closest warehouse on a straight line because roads, traffic, and rivers get in the way.

This notebook covers:
1. Finding nearby DCs using the **Haversine formula** (distance on a sphere)
2. Filtering candidates with **delivery zones**
3. Estimating delivery time from historical data
4. Caching nearby-DC lookups in Redis for speed

## Learning Goals

- Calculate real distances between lat/lon points (not just Euclidean)
- Understand the **two-step** nearby DC lookup: rough filter → precise estimate
- Build an ETA model from historical order data
- See how caching spatial lookups keeps latency under 100 ms

## 🛠️ Setup

```bash
cd system-designs/gopuff
docker-compose up -d
```

### Kernel Selection (VS Code)
Select the `.venv` kernel from the kernel picker (top-right).
If it doesn’t appear, reload VS Code (`Cmd+Shift+P` → “Reload Window”).

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import math
import time

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "gopuff", "user": "demo", "password": "demo",
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

conn = get_db_connection()
r = get_redis_client()
r.ping()
print("✅ Postgres and Redis connected")
conn.close()

## 1️⃣ The Haversine Formula — Distance on Earth

Earth is (roughly) a sphere. The straight-line distance between two GPS points on a flat map is **wrong** — it ignores the curvature. The **Haversine formula** calculates the great-circle distance (shortest path along the surface).

### Why not just use `sqrt((x2-x1)² + (y2-y1)²)`?

Euclidean distance treats latitude and longitude like a flat grid. Near the equator that’s roughly OK, but:
- 1° of longitude = 69 miles at the equator, but only 49 miles in Austin, TX
- At the poles, 1° of longitude = 0 miles!

The Haversine formula handles this correctly.

In [ ]:
def haversine_miles(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two points
    on Earth (in miles) using the Haversine formula.
    """
    R = 3958.8  # Earth's radius in miles
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + \
        math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    c = 2 * math.asin(math.sqrt(a))
    return R * c


dist = haversine_miles(30.2672, -97.7431, 30.3716, -97.7064)
print(f"DC Downtown → DC North: {dist:.2f} miles")

euclidean = math.sqrt((30.3716 - 30.2672)**2 + (-97.7064 - (-97.7431))**2) * 69
print(f"Euclidean estimate:     {euclidean:.2f} miles")
print(f"Difference:             {abs(dist - euclidean):.2f} miles")

## 2️⃣ Finding Nearby DCs

The interview solution uses a **two-step approach**:

1. **Rough filter**: Find all DCs within a generous radius (e.g., 60 miles). This is fast because we just compute distance.
2. **Precise filter**: For the candidates from step 1, check the actual delivery zones and estimated drive times.

This avoids calling an expensive travel-time API for DCs that are clearly too far away.

In [ ]:
def find_nearby_dcs(customer_lat, customer_lon, max_radius_miles=10.0):
    """Find all DCs within max_radius_miles of the customer using Haversine."""
    conn = get_db_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("SELECT id, name, latitude, longitude FROM distribution_centers WHERE is_active = TRUE")
    all_dcs = cur.fetchall()
    conn.close()

    nearby = []
    for dc in all_dcs:
        dist = haversine_miles(customer_lat, customer_lon, dc["latitude"], dc["longitude"])
        if dist <= max_radius_miles:
            nearby.append({"dc_id": dc["id"], "name": dc["name"],
                           "distance_miles": round(dist, 2),
                           "lat": dc["latitude"], "lon": dc["longitude"]})
    nearby.sort(key=lambda x: x["distance_miles"])
    return nearby


customer_lat, customer_lon = 30.2850, -97.7335
nearby = find_nearby_dcs(customer_lat, customer_lon, max_radius_miles=8)

print(f"📍 Customer at ({customer_lat}, {customer_lon})")
print(f"   Found {len(nearby)} DCs within 8 miles:\n")
for dc in nearby:
    bar = "█" * int(dc['distance_miles'] * 3)
    print(f"   {dc['name']:<18s} {dc['distance_miles']:5.2f} mi  {bar}")

## 3️⃣ Delivery Zones and ETA Estimation

Each DC has **delivery zones** — concentric rings with different delivery time estimates. A DC’s “core” zone (3 miles) might promise 15-minute delivery, while the “extended” zone (7 miles) takes 30 minutes.

### How ETA Works

```
Customer Location
       │
       ▼
  Find nearby DCs (Haversine)
       │
       ▼
  For each DC, check delivery zones
       │
       ├── Distance < core radius? → Use core ETA
       ├── Distance < extended radius? → Use extended ETA
       └── Too far? → Skip this DC
       │
       ▼
  Pick the DC with the best (lowest) ETA
```

In [ ]:
def estimate_delivery(customer_lat, customer_lon):
    """For each nearby DC, find matching delivery zone and estimate time."""
    candidates = find_nearby_dcs(customer_lat, customer_lon, max_radius_miles=15)
    if not candidates:
        return []

    conn = get_db_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    dc_ids = [c["dc_id"] for c in candidates]
    cur.execute("""
        SELECT dc_id, zone_name, max_radius_miles, avg_delivery_minutes, surge_multiplier
        FROM delivery_zones WHERE dc_id = ANY(%s)
        ORDER BY dc_id, max_radius_miles
    """, (dc_ids,))
    zones = cur.fetchall()
    conn.close()

    zone_map = {}
    for z in zones:
        zone_map.setdefault(z["dc_id"], []).append(z)

    results = []
    for cand in candidates:
        dc_zones = zone_map.get(cand["dc_id"], [])
        for z in dc_zones:
            if cand["distance_miles"] <= z["max_radius_miles"]:
                factor = cand["distance_miles"] / z["max_radius_miles"]
                est_min = int(z["avg_delivery_minutes"] * (0.7 + 0.6 * factor))
                results.append({
                    "dc_id": cand["dc_id"], "dc_name": cand["name"],
                    "distance_miles": cand["distance_miles"],
                    "zone": z["zone_name"],
                    "estimated_minutes": est_min,
                    "surge_multiplier": float(z["surge_multiplier"]),
                })
                break

    results.sort(key=lambda x: x["estimated_minutes"])
    return results


estimates = estimate_delivery(30.2850, -97.7335)
print("🚗 Delivery Estimates")
print("=" * 70)
for e in estimates:
    clk = "🟢" if e["estimated_minutes"] <= 20 else "🟡" if e["estimated_minutes"] <= 35 else "🔴"
    print(f"  {clk} {e['dc_name']:<18s} | {e['distance_miles']:5.2f} mi | "
          f"{e['zone']:<22s} | ~{e['estimated_minutes']} min")

## 4️⃣ Improving ETA with Historical Data

Our zone-based estimate is a starting point, but real delivery times depend on:
- **Time of day** — lunch rush vs late night
- **Day of week** — weekends are busier
- **Actual past deliveries** — real data beats estimates

Let’s use our historical order data to build a better ETA model.

In [ ]:
def get_historical_eta(dc_id, hour_of_day):
    """Look at past delivered orders to estimate delivery time by hour."""
    conn = get_db_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT
            COUNT(*) AS sample_size,
            ROUND(AVG(actual_delivery_minutes)) AS avg_minutes,
            ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY actual_delivery_minutes)) AS median_minutes,
            ROUND(PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY actual_delivery_minutes)) AS p90_minutes,
            MIN(actual_delivery_minutes) AS min_minutes,
            MAX(actual_delivery_minutes) AS max_minutes
        FROM orders
        WHERE dc_id = %s AND EXTRACT(HOUR FROM created_at) = %s
          AND status = 'delivered' AND actual_delivery_minutes IS NOT NULL
    """, (dc_id, hour_of_day))
    result = cur.fetchone()
    conn.close()
    return dict(result) if result else {}


print("📊 Historical Delivery Times — DC Downtown")
print("=" * 60)
print(f"  {'Hour':<8s} {'Samples':>8s} {'Avg':>6s} {'Median':>8s} {'P90':>6s}")
print("-" * 60)
for hour in [8, 12, 15, 18, 21]:
    stats = get_historical_eta(dc_id=1, hour_of_day=hour)
    if stats and stats["sample_size"] > 0:
        print(f"  {hour:02d}:00   {stats['sample_size']:>8d} {stats['avg_minutes']:>5.0f}m "
              f"{stats['median_minutes']:>7.0f}m {stats['p90_minutes']:>5.0f}m")
    else:
        print(f"  {hour:02d}:00   {'no data':>8s}")

## 5️⃣ Caching the Nearby DC Lookup

Every availability request starts with “find nearby DCs.” Since DCs don’t move (they’re buildings!), we can cache this aggressively.

**Strategy**: Round the customer’s lat/lon to a grid cell (e.g., 0.01° ≈ 0.7 miles) and cache the result. Customers in the same grid cell get the same nearby DCs.

In [ ]:
def find_nearby_dcs_cached(customer_lat, customer_lon, max_radius_miles=10.0, grid_precision=2):
    """Cached nearby-DC lookup with grid-cell rounding."""
    r = get_redis_client()
    grid_lat = round(customer_lat, grid_precision)
    grid_lon = round(customer_lon, grid_precision)
    cache_key = f"nearby:{grid_lat},{grid_lon}:{max_radius_miles}"

    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), "cache"

    nearby = find_nearby_dcs(customer_lat, customer_lon, max_radius_miles)
    r.setex(cache_key, 300, json.dumps(nearby))
    return nearby, "database"


r = get_redis_client()
r.flushdb()

start = time.time()
nearby, source = find_nearby_dcs_cached(30.2850, -97.7335)
t1 = (time.time() - start) * 1000
print(f"1st call: {source:>8s} | {t1:.1f} ms | {len(nearby)} DCs")

start = time.time()
nearby, source = find_nearby_dcs_cached(30.2850, -97.7335)
t2 = (time.time() - start) * 1000
print(f"2nd call: {source:>8s} | {t2:.1f} ms | {len(nearby)} DCs")

start = time.time()
nearby, source = find_nearby_dcs_cached(30.2855, -97.7330)
t3 = (time.time() - start) * 1000
print(f"Near loc: {source:>8s} | {t3:.1f} ms | {len(nearby)} DCs  (same grid cell!)")

## 6️⃣ Visualizing DC Coverage

Let’s build a simple text-based map to see how our DCs cover the Austin metro area.

In [ ]:
conn = get_db_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT id, name, latitude, longitude FROM distribution_centers ORDER BY id")
dcs = cur.fetchall()
conn.close()

print("\n🗺️  Austin DC Map (text-based)")
print("   North ↑\n")

min_lat = min(d['latitude'] for d in dcs) - 0.02
max_lat = max(d['latitude'] for d in dcs) + 0.02
min_lon = min(d['longitude'] for d in dcs) - 0.02
max_lon = max(d['longitude'] for d in dcs) + 0.02

ROWS, COLS = 20, 50
grid = [[' ' for _ in range(COLS)] for _ in range(ROWS)]

for dc in dcs:
    row = ROWS - 1 - int((dc['latitude'] - min_lat) / (max_lat - min_lat) * (ROWS - 1))
    col = int((dc['longitude'] - min_lon) / (max_lon - min_lon) * (COLS - 1))
    row = max(0, min(ROWS - 1, row))
    col = max(0, min(COLS - 1, col))
    grid[row][col] = str(dc['id'])

cr = ROWS - 1 - int((30.2850 - min_lat) / (max_lat - min_lat) * (ROWS - 1))
cc = int((-97.7335 - min_lon) / (max_lon - min_lon) * (COLS - 1))
grid[max(0, min(ROWS-1, cr))][max(0, min(COLS-1, cc))] = '★'

for row in grid:
    print('   │' + ''.join(row) + '│')

print("\n   Legend: ★ = Customer")
for dc in dcs:
    print(f"           {dc['id']} = {dc['name']}")

## 🔑 Key Takeaways

| Concept | What We Did |
|---------|-------------|
| **Haversine formula** | Accurate distance between lat/lon points on Earth’s surface |
| **Two-step lookup** | Rough distance filter → precise zone-based check |
| **Delivery zones** | Pre-computed rings around each DC with estimated times |
| **Historical ETA** | Used past order data to refine time-of-day estimates |
| **Grid-based caching** | Round lat/lon so nearby customers share cache entries |

### Interview Tip

When discussing nearby-DC lookup, mention the **optimization progression**:
1. Simple Euclidean distance (bad — ignores Earth’s curvature)
2. Haversine + fixed radius (good — accurate distance)
3. Haversine + external travel time API for candidates (great — accounts for roads/traffic)

Also note that DCs are **static** (buildings don’t move), so the nearby-DC result can be cached aggressively.

## 🧹 Cleanup

In [ ]:
r = get_redis_client()
keys = r.keys("nearby:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")